In [ ]:
import sys

sys.path.append("..")

In [ ]:
from tqdm import tqdm
import numpy as np
import os
from matplotlib import pyplot as plt
from os.path import join as pjoin
from omegaconf import OmegaConf

import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset

from utils.misc import load_config
from engine.policy import train
from datasets.data_preparation import prepare_data
from utils.misc import plot_loss_curve, current_time, load_policy_encoder_from_pretraining
from model.vit import build_model

In [ ]:
exp_path = '../results/policy/dl.vit/2023-08-08 12Hr 53Min 41Sec IST+0530'
encoder_checkpoint_path = '../pretraining/encoder/dl.vit/2023-08-21 18Hr 09Min 32Sec IST+0530/checkpoint.pt'

config = load_config('.', exp_path, 'hyperparameters.yaml')

In [ ]:
trainer_config = config['trainer']
data_config = config['data']

checkpoint_name = trainer_config['checkpoint_name']
device = trainer_config['device']
batch_size = trainer_config['batch_size']

patch_size = data_config['patch']['patch_size']
lithology_classes = data_config['lithology_classes']

config['root'] = '..'

In [ ]:
x_train, x_val, y_train, y_val, num_classes = prepare_data(config)

In [ ]:
traindataset = TensorDataset(x_train, y_train)
trainloader = DataLoader(traindataset, batch_size=batch_size, shuffle=True)

valdataset = TensorDataset(x_val, y_val)
valloader = DataLoader(valdataset, batch_size=batch_size, shuffle=False)

# invert key as value and value as key
lithology_names = {v: k for k, v in lithology_classes.items()}

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
config['trainer']['device'] = device

In [ ]:
config['model']['dim'] = 150

model = build_model(config)

if encoder_checkpoint_path is not None:
    model = load_policy_encoder_from_pretraining(model, encoder_checkpoint_path, device)

In [ ]:
model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
(
    train_losses, 
    val_losses, 
    train_accuracies, 
    val_accuracies, 
    best_epoch, 
    best_loss, 
    best_cm_val, 
    best_cm, 
    best_model_chkpt, 
    best_optim_chkpt
) = train(num_epochs=100,
          model=model,
          train_loader=trainloader,
          val_loader=valloader,
          criterion=criterion,
          optimizer=optimizer,
          tolerance=config['callbacks']['early_stopping_tolerance'],
          device=device)

In [ ]:
plt.plot(train_accuracies)
plt.plot(val_accuracies)